In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# DRAFT 2: GATED HYBRID PHISHING EMAIL MODEL
# TRAIN + SAVE TO COLAB + GOOGLE DRIVE + LOCAL DEVICE
# ============================================================

# -----------------------------
# 1. Libraries
# -----------------------------
import os
import re
import json
import shutil
import zipfile

import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

import torch
import torch.nn as nn

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

# Google Drive
from google.colab import drive
from google.colab import files


# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount("/content/drive")

DRIVE_SAVE_DIR = "/content/drive/MyDrive/Draft2_Phishing_Model"
COLAB_SAVE_DIR = "/content/Draft2_Phishing_Model"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
os.makedirs(COLAB_SAVE_DIR, exist_ok=True)

print("Google Drive mounted.")
print("Colab save directory:", COLAB_SAVE_DIR)
print("Drive save directory:", DRIVE_SAVE_DIR)


# -----------------------------
# 3. Load Dataset
# -----------------------------
train_df = pd.read_csv("/content/phishing_train.csv")
val_df = pd.read_csv("/content/phishing_validation.csv")
test_df = pd.read_csv("/content/phishing_test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# -----------------------------
# 4. NLP Feature Definitions
# -----------------------------
URGENCY_WORDS = {
    "urgent", "urgently", "immediately", "immediate",
    "now", "today", "asap", "quickly", "deadline",
    "expire", "expired", "final", "action"
}

CREDENTIAL_WORDS = {
    "password", "passwd", "username", "login",
    "credential", "credentials", "verify", "verification",
    "authenticate", "authentication", "account"
}

THREAT_WORDS = {
    "suspend", "suspended", "terminate", "terminated",
    "blocked", "block", "close", "closed", "penalty",
    "fraud", "unauthorized", "warning", "security"
}

FINANCIAL_WORDS = {
    "payment", "pay", "invoice", "money", "bank",
    "transfer", "transaction", "refund", "credit",
    "debit", "fee", "account", "billing"
}

CTA_WORDS = {
    "click", "clicking", "visit", "open", "download",
    "confirm", "verify", "submit", "update",
    "activate", "login"
}


def count_terms(text, vocabulary):
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    return sum(word in vocabulary for word in words)


def extract_nlp_features(text):

    text = str(text)

    words = re.findall(r"\b[a-zA-Z]+\b", text)

    word_count = max(len(words), 1)

    uppercase_words = [
        word for word in words
        if len(word) > 1 and word.isupper()
    ]

    uppercase_ratio = len(uppercase_words) / word_count

    url_count = len(
        re.findall(
            r"https?://\S+|www\.\S+",
            text,
            flags=re.IGNORECASE
        )
    )

    email_count = len(
        re.findall(
            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
            text
        )
    )

    phone_count = len(
        re.findall(
            r"(?:\+?\d[\d\s().-]{7,}\d)",
            text
        )
    )

    features = [
        count_terms(text, URGENCY_WORDS),
        count_terms(text, CREDENTIAL_WORDS),
        count_terms(text, THREAT_WORDS),
        count_terms(text, FINANCIAL_WORDS),
        count_terms(text, CTA_WORDS),
        url_count,
        email_count,
        phone_count,
        text.count("!"),
        text.count("?"),
        uppercase_ratio,
        np.log1p(len(text))
    ]

    return np.array(features, dtype=np.float32)


# -----------------------------
# 5. Generate NLP Features
# -----------------------------
train_features = np.vstack(
    train_df["processed_text"].apply(extract_nlp_features)
)

val_features = np.vstack(
    val_df["processed_text"].apply(extract_nlp_features)
)

test_features = np.vstack(
    test_df["processed_text"].apply(extract_nlp_features)
)

print("Feature shape:", train_features.shape)
print("First feature vector:")
print(train_features[0])


# -----------------------------
# 6. Feature Normalisation
# -----------------------------
feature_mean = train_features.mean(axis=0)
feature_std = train_features.std(axis=0)

feature_std[feature_std == 0] = 1.0

train_features = (
    (train_features - feature_mean) / feature_std
)

val_features = (
    (val_features - feature_mean) / feature_std
)

test_features = (
    (test_features - feature_mean) / feature_std
)

print("Feature normalisation complete.")


# -----------------------------
# 7. DeBERTa Tokenisation
# -----------------------------
MODEL_NAME = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 512


def tokenize_function(examples):
    return tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )


# -----------------------------
# 8. Convert to Hugging Face Dataset
# -----------------------------
train_dataset = Dataset.from_pandas(
    train_df[["processed_text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["processed_text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["processed_text", "label"]],
    preserve_index=False
)


# -----------------------------
# 9. Add NLP Features
# -----------------------------
train_dataset = train_dataset.add_column(
    "nlp_features",
    train_features.tolist()
)

val_dataset = val_dataset.add_column(
    "nlp_features",
    val_features.tolist()
)

test_dataset = test_dataset.add_column(
    "nlp_features",
    test_features.tolist()
)


# -----------------------------
# 10. Tokenise Dataset
# -----------------------------
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

print("Tokenisation complete.")


# -----------------------------
# 11. Draft 2 Model
# -----------------------------
class GatedHybridPhishingModel(nn.Module):

    def __init__(self, model_name, num_features=12, num_labels=2):
        super().__init__()

        # Pretrained DeBERTa backbone
        self.transformer = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32
        )

        hidden_size = self.transformer.config.hidden_size

        # Context projection
        self.context_projection = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # NLP feature projection
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Learnable gate
        self.gate = nn.Sequential(
            nn.Linear(512, 256),
            nn.Sigmoid()
        )

        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Final classifier
        self.classifier = nn.Linear(256, num_labels)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        nlp_features=None,
        labels=None
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # First-token contextual representation
        contextual_embedding = outputs.last_hidden_state[:, 0, :]

        # Context branch
        context = self.context_projection(
            contextual_embedding
        )

        # Explicit NLP feature branch
        features = self.feature_projection(
            nlp_features.float()
        )

        # Concatenate both branches for gate calculation
        gate_input = torch.cat(
            [context, features],
            dim=1
        )

        # Learnable gate
        gate = self.gate(gate_input)

        # Gated fusion
        fused = (
            gate * context
            + (1 - gate) * features
        )

        # Final fusion processing
        fused = self.fusion(fused)

        # Classification
        logits = self.classifier(fused)

        loss = None

        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }


# -----------------------------
# 12. Create Model
# -----------------------------
model = GatedHybridPhishingModel(
    MODEL_NAME,
    num_features=12,
    num_labels=2
)

model.transformer.gradient_checkpointing_enable()
model.transformer.config.use_cache = False

print("Draft 2 model ready.")


# -----------------------------
# 13. Check Model Parameters
# -----------------------------
nan_params = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        nan_params.append(name)

print("NaN/Inf parameters:", len(nan_params))


# -----------------------------
# 14. Data Collator
# -----------------------------
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# -----------------------------
# 15. Evaluation Metrics
# -----------------------------
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# -----------------------------
# 16. Training Arguments
# -----------------------------
training_args = TrainingArguments(

    output_dir="/content/hybrid_phishing_draft2_results",

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    weight_decay=0.01,

    warmup_steps=263,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,

    optim="adamw_torch",

    seed=42,
    data_seed=42,

    remove_unused_columns=False,

    report_to="none",

    save_total_limit=2
)


# -----------------------------
# 17. Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Draft 2 Trainer ready.")


# -----------------------------
# 18. Train
# -----------------------------
print("\nStarting Draft 2 training...\n")

train_result = trainer.train()

print("\nTraining completed.")


# -----------------------------
# 19. Test Evaluation
# -----------------------------
test_results = trainer.evaluate(
    test_tokenized,
    metric_key_prefix="test"
)

print("\nTest Results:")
print(test_results)


# -----------------------------
# 20. Classification Report
# -----------------------------
predictions = trainer.predict(
    test_tokenized
)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Safe Email",
            "Phishing Email"
        ],
        digits=4
    )
)

print("Confusion Matrix:")

cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)


# ============================================================
# 21. SAVE EVERYTHING
# ============================================================

print("\n" + "=" * 60)
print("SAVING DRAFT 2 MODEL")
print("=" * 60)


# -----------------------------
# 21A. Save to Colab
# -----------------------------
colab_model_dir = os.path.join(
    COLAB_SAVE_DIR,
    "model"
)

os.makedirs(
    colab_model_dir,
    exist_ok=True
)

# Save model weights
torch.save(
    trainer.model.state_dict(),
    os.path.join(
        colab_model_dir,
        "pytorch_model.bin"
    )
)

# Save tokenizer
tokenizer.save_pretrained(
    colab_model_dir
)

# Save transformer configuration
trainer.model.transformer.config.save_pretrained(
    colab_model_dir
)

# Save custom model information
model_config = {
    "model_name": MODEL_NAME,
    "num_features": 12,
    "num_labels": 2,
    "max_length": MAX_LENGTH,
    "context_dimension": trainer.model.transformer.config.hidden_size,
    "context_projection_dimension": 256,
    "feature_projection_dimension": 256,
    "fusion_dimension": 256,
    "architecture": "DeBERTa-v3-large + 12 NLP features + gated fusion",
    "random_seed": 42
}

with open(
    os.path.join(
        colab_model_dir,
        "model_config.json"
    ),
    "w"
) as f:
    json.dump(
        model_config,
        f,
        indent=4
    )

# Save feature normalisation values
np.save(
    os.path.join(
        colab_model_dir,
        "feature_mean.npy"
    ),
    feature_mean
)

np.save(
    os.path.join(
        colab_model_dir,
        "feature_std.npy"
    ),
    feature_std
)

# Save feature names
feature_names = [
    "urgency_term_count",
    "credential_term_count",
    "threat_term_count",
    "financial_term_count",
    "cta_term_count",
    "url_count",
    "email_count",
    "phone_count",
    "exclamation_count",
    "question_count",
    "uppercase_word_ratio",
    "log_text_length"
]

with open(
    os.path.join(
        colab_model_dir,
        "feature_names.json"
    ),
    "w"
) as f:
    json.dump(
        feature_names,
        f,
        indent=4
    )

print("Saved to Colab:")
print(colab_model_dir)


# -----------------------------
# 21B. Copy complete model to Google Drive
# -----------------------------
drive_model_dir = os.path.join(
    DRIVE_SAVE_DIR,
    "model"
)

if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)

shutil.copytree(
    colab_model_dir,
    drive_model_dir
)

print("Saved to Google Drive:")
print(drive_model_dir)


# -----------------------------
# 21C. Save test results
# -----------------------------
results = {
    "test_results": {
        k: float(v)
        for k, v in test_results.items()
        if isinstance(v, (int, float, np.integer, np.floating))
    },
    "confusion_matrix": cm.tolist()
}

with open(
    os.path.join(
        colab_model_dir,
        "test_results.json"
    ),
    "w"
) as f:
    json.dump(
        results,
        f,
        indent=4
    )

# Also copy updated results to Drive
shutil.copy(
    os.path.join(
        colab_model_dir,
        "test_results.json"
    ),
    os.path.join(
        drive_model_dir,
        "test_results.json"
    )
)


# ============================================================
# 22. CREATE ZIP FOR LOCAL COMPUTER
# ============================================================

zip_path = "/content/Draft2_Phishing_Model.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files_list in os.walk(
        COLAB_SAVE_DIR
    ):

        for file_name in files_list:

            file_path = os.path.join(
                root,
                file_name
            )

            archive_name = os.path.relpath(
                file_path,
                COLAB_SAVE_DIR
            )

            zipf.write(
                file_path,
                archive_name
            )


print("\nZIP created:")
print(zip_path)


# ============================================================
# 23. DOWNLOAD TO LOCAL COMPUTER
# ============================================================

print("\nDownloading model to your local computer...")

files.download(
    zip_path
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("DRAFT 2 MODEL SAVING COMPLETE")
print("=" * 60)

print("\n1. COLAB:")
print(COLAB_SAVE_DIR)

print("\n2. GOOGLE DRIVE:")
print(DRIVE_SAVE_DIR)

print("\n3. LOCAL COMPUTER:")
print("Draft2_Phishing_Model.zip")

print("\nThe saved package contains:")
print("- DeBERTa-v3-large weights")
print("- Custom GatedHybridPhishingModel information")
print("- Tokenizer")
print("- Transformer configuration")
print("- Feature mean")
print("- Feature standard deviation")
print("- 12 feature names")
print("- Model configuration")
print("- Test results")

Mounted at /content/drive
Google Drive mounted.
Colab save directory: /content/Draft2_Phishing_Model
Drive save directory: /content/drive/MyDrive/Draft2_Phishing_Model
Train: (14015, 4)
Validation: (1752, 4)
Test: (1752, 4)
Feature shape: (14015, 12)
First feature vector:
[0.       0.       1.       0.       0.       0.       0.       0.
 0.       1.       0.       6.368187]
Feature normalisation complete.


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...


Test Results:
{'test_loss': 0.03715737536549568, 'test_accuracy': 0.9948630136986302, 'test_precision': 0.9969183359013868, 'test_recall': 0.9892966360856269, 'test_f1': 0.9930928626247122}



Classification Report:
                precision    recall  f1-score   support

    Safe Email     0.9937    0.9982    0.9959      1098
Phishing Email     0.9969    0.9893    0.9931       654

      accuracy                         0.9949      1752
     macro avg     0.9953    0.9937    0.9945      1752
  weighted avg     0.9949    0.9949    0.9949      1752

Confusion Matrix:
[[1096    2]
 [   7  647]]

SAVING DRAFT 2 MODEL
Saved to Colab:
/content/Draft2_Phishing_Model/model
Saved to Google Drive:
/content/drive/MyDrive/Draft2_Phishing_Model/model

ZIP created:
/content/Draft2_Phishing_Model.zip



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DRAFT 2 MODEL SAVING COMPLETE

1. COLAB:
/content/Draft2_Phishing_Model

2. GOOGLE DRIVE:
/content/drive/MyDrive/Draft2_Phishing_Model

3. LOCAL COMPUTER:
Draft2_Phishing_Model.zip

The saved package contains:
- DeBERTa-v3-large weights
- Custom GatedHybridPhishingModel information
- Tokenizer
- Transformer configuration
- Feature mean
- Feature standard deviation
- 12 feature names
- Model configuration
- Test results


In [ ]:
# Save newly needed files for later LIME/SHAP + Streamlit + Browser Extension

import os
import json

SAVE_DIR = "/content/Draft2_Phishing_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save the exact custom model architecture
model_code = '''
import torch
import torch.nn as nn
from transformers import AutoModel

class GatedHybridPhishingModel(nn.Module):

    def __init__(self, model_name, num_features=12, num_labels=2):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32
        )

        hidden_size = self.transformer.config.hidden_size

        self.context_projection = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.gate = nn.Sequential(
            nn.Linear(512, 256),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(256, num_labels)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        nlp_features=None,
        labels=None
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        contextual_embedding = outputs.last_hidden_state[:, 0, :]

        context = self.context_projection(
            contextual_embedding
        )

        features = self.feature_projection(
            nlp_features.float()
        )

        gate_input = torch.cat(
            [context, features],
            dim=1
        )

        gate = self.gate(gate_input)

        fused = (
            gate * context
            + (1 - gate) * features
        )

        fused = self.fusion(fused)

        logits = self.classifier(fused)

        loss = None

        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }
'''

with open(
    os.path.join(SAVE_DIR, "GatedHybridPhishingModel.py"),
    "w"
) as f:
    f.write(model_code)


# 2. Save the exact 12 feature names
feature_names = [
    "urgency_term_count",
    "credential_term_count",
    "threat_term_count",
    "financial_term_count",
    "cta_term_count",
    "url_count",
    "email_count",
    "phone_count",
    "exclamation_count",
    "question_count",
    "uppercase_word_ratio",
    "log_text_length"
]

with open(
    os.path.join(SAVE_DIR, "feature_names.json"),
    "w"
) as f:
    json.dump(feature_names, f, indent=4)


# 3. Save the feature normalisation values
np.save(
    os.path.join(SAVE_DIR, "feature_mean.npy"),
    feature_mean
)

np.save(
    os.path.join(SAVE_DIR, "feature_std.npy"),
    feature_std
)


# 4. Save model configuration
model_config = {
    "model_name": "microsoft/deberta-v3-large",
    "num_features": 12,
    "num_labels": 2,
    "max_length": 512,
    "context_projection": 256,
    "feature_projection": 256,
    "fusion_dimension": 256,
    "labels": {
        "0": "Safe Email",
        "1": "Phishing Email"
    }
}

with open(
    os.path.join(SAVE_DIR, "model_config.json"),
    "w"
) as f:
    json.dump(model_config, f, indent=4)


# 5. Save the exact preprocessing function
preprocessing_code = '''
import re
import html
import unicodedata

def preprocess_text(text):

    if text is None:
        return ""

    text = str(text)

    text = html.unescape(text)

    text = unicodedata.normalize("NFKC", text)

    text = "".join(
        char for char in text
        if unicodedata.category(char) != "Cc"
        or char in "\\n\\t"
    )

    text = re.sub(r"<[^>]+>", " ", text)

    text = re.sub(r"\\r\\n|\\r|\\n", " ", text)

    text = re.sub(r"\\s+", " ", text).strip()

    return text
'''

with open(
    os.path.join(SAVE_DIR, "preprocessing.py"),
    "w"
) as f:
    f.write(preprocessing_code)


# 6. Create a requirements file for future deployment
requirements = """torch
transformers
numpy
pandas
scikit-learn
streamlit
lime
shap
"""

with open(
    os.path.join(SAVE_DIR, "requirements.txt"),
    "w"
) as f:
    f.write(requirements)


print("New deployment files saved to:")
print(SAVE_DIR)
print("\nFiles created:")
print("- GatedHybridPhishingModel.py")
print("- feature_names.json")
print("- feature_mean.npy")
print("- feature_std.npy")
print("- model_config.json")
print("- preprocessing.py")
print("- requirements.txt")

New deployment files saved to:
/content/Draft2_Phishing_Model

Files created:
- GatedHybridPhishingModel.py
- feature_names.json
- feature_mean.npy
- feature_std.npy
- model_config.json
- preprocessing.py
- requirements.txt


In [ ]:
import subprocess
import sys

with open("/content/Draft2_Phishing_Model/installed_packages.txt", "w") as f:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=f,
        text=True
    )

print("✓ Exact installed package versions saved.")

✓ Exact installed package versions saved.
